# CSCI E-89 LLecture 11
## RAG with Ollama, LangChain and a local DeepSeek-r1 model
### Lakshmi Umarale

**Learning objectives.** By the end of this lab you will be able to:

1. Install **Ollama**, download a model, and run it from the command line
2. Explain what `ollama list` and `ollama ps` tell you, and why reasoning models feel slow
3. Customize a model with a **Modelfile**
4. Call a local model through the **REST API** (`curl`) and the **Python library**, and separate its *chain of thought* from its *final answer*
5. Build a **RAG application** that answers questions about an uploaded PDF, using LangChain, Chroma, DeepSeek-R1 and a Gradio interface
6. Recognise common RAG pitfalls (documents mixing together, repeated re-embedding, hallucinated answers) and how to avoid them

## Table Of Contents
* [Part 1: Running models locally with Ollama](#part1)
  * [Install Ollama](#install)
  * [Choose a model size](#sizes)
  * [Step 1: Inference from the command line](#cli)
  * [Customize a model with a Modelfile](#modelfile)
  * [Step 2: Accessing the model through the API](#api)
  * [Step 3: Accessing the model through Python](#python)
* [Part 2: A local RAG app with Gradio](#part2)
  * [Prerequisites and imports](#prereq)
  * [Step 1: Process the PDF](#process)
  * [Step 2: Combine the retrieved chunks](#combine)
  * [Step 3: Query DeepSeek-R1](#query)
  * [Step 4: The RAG pipeline](#pipeline)
  * [Step 5: The Gradio interface](#gradio)
* [Troubleshooting](#trouble)
* [Summary and exercises](#summary)

# Part 1: Running models locally with Ollama <a id="part1"></a>

**Ollama** simplifies running large language models on your own computer. It handles model downloads, quantization (shrinking a model so it fits in memory) and execution. Your prompts never leave your machine, and there is no API key or per-request cost.

### Install Ollama <a id="install"></a>

- **macOS / Windows:** download and run the installer from <https://ollama.com/download>
- **Linux:** run this command in a terminal:

```bash
curl -fsSL https://ollama.com/install.sh | bash
```

Check that it works (the Ollama app or service must be running):

```bash
ollama --version
ollama list
```

`ollama list` shows the models already downloaded to your computer, for example:

```text
NAME                       ID              SIZE
llama3.2:latest            a80c4f17acd5    2.0 GB
nomic-embed-text:latest    0a109f422b47    274 MB
deepseek-r1:8b             6995872bfe4c    5.2 GB
deepseek-r1:latest         6995872bfe4c    5.2 GB
```

Notice that `deepseek-r1:8b` and `deepseek-r1:latest` have the **same ID**: a model name without a tag means `:latest`, which here is the 8B version.

Useful commands:

| Command | What it does |
|---|---|
| `ollama pull <model>` | download a model |
| `ollama list` | list downloaded models |
| `ollama run <model>` | chat with a model in the terminal |
| `ollama ps` | show models currently loaded in memory |
| `ollama stop <model>` | unload a model from memory |
| `ollama rm <model>` | delete a model from disk |

### Choose a model size <a id="sizes"></a>

Ollama offers a range of DeepSeek-R1 models, from 1.5B parameters to the full 671B model. The 671B model is the original DeepSeek-R1. The smaller ones are **distilled** versions built on Qwen and Llama architectures. If your hardware cannot support the 671B model (almost no laptop can), run a smaller one by replacing `X` below with a size (`1.5b`, `7b`, `8b`, `14b`, `32b`, `70b`, `671b`):

```bash
ollama pull deepseek-r1:Xb
```

As a rule of thumb, you need **at least as much free memory (RAM or GPU memory) as the size of the download**, plus extra for the context window. On a laptop, start with `deepseek-r1:1.5b` or `deepseek-r1:8b`.

### Step 1: Inference from the command line <a id="cli"></a>

```bash
ollama run deepseek-r1
```

DeepSeek-R1 is a **reasoning model**: it first writes a long "thinking" section, then the final answer. A shortened session looks like this:

```text
>>> What is the capital of France?
Thinking...
Okay, the user is asking for the capital of France. Let me start by recalling
the basic answer. Paris is indeed the capital, but I should make sure it's
still current. [...]
...done thinking.

The capital of France is **Paris**.

It is the largest city in France and the seat of the French government. [...]

>>> Send a message (/? for help)
```

Type `/bye` to leave the session. While a model is running, check what is loaded in memory:

```bash
ollama ps
```

```text
NAME                  ID              SIZE      PROCESSOR    CONTEXT    UNTIL
deepseek-r1:latest    6995872bfe4c    9.9 GB    100% GPU     32768      About a minute from now
```

How to read it:

- **SIZE** (9.9 GB) is larger than the download (5.2 GB) because it includes memory for the context window.
- **PROCESSOR**: `100% GPU` means the whole model fits in GPU memory, so it is fast. If you see a large `CPU` share, the model did not fit and generation will be much slower.
- **CONTEXT** (32768): how many tokens of text the model can look at.
- **UNTIL**: when Ollama will unload the model to free memory (about 5 minutes after the last use by default).

### Customize a model with a Modelfile <a id="modelfile"></a>

A **Modelfile** is a small text file that starts from an existing model and changes its settings. Create a file named `Modelfile` containing:

```text
FROM deepseek-r1
PARAMETER temperature 0.6
PARAMETER num_ctx 8192
SYSTEM You are a patient teaching assistant. Explain your answers step by step.
```

- `FROM`: the base model
- `PARAMETER`: a setting, such as `temperature` (randomness), `top_p`, `top_k`, `repeat_penalty` or `num_ctx` (context size)
- `SYSTEM`: a system prompt that is applied to every conversation

Then create and run your custom model:

```bash
ollama create deepseek-teacher -f Modelfile
ollama run deepseek-teacher
```

> A larger `num_ctx` lets the model read longer inputs but uses more memory.

### Step 2: Accessing the model through the API <a id="api"></a>

Ollama runs a local web server on port **11434**. To integrate a model into an application, send it HTTP requests, for example with `curl`.

List the downloaded models (the reply is JSON with each model's name, size, parameter count and capabilities, such as `completion`, `thinking`, `tools` or `embedding`):

```bash
curl http://localhost:11434/api/tags
```

Send a chat request:

```bash
curl http://localhost:11434/api/chat -d '{
  "model": "deepseek-r1",
  "messages": [{ "role": "user", "content": "Solve: 25 * 25" }],
  "stream": false
}'
```

The reply is a single JSON object (shortened here):

```json
{
  "model": "deepseek-r1",
  "message": {
    "role": "assistant",
    "content": "To solve 25 x 25 ... \\boxed{625}",
    "thinking": "I need to solve 25 times 25. That's straightforward multiplication. ..."
  },
  "done": true,
  "total_duration": 121081087458,
  "eval_count": 2614
}
```

What to notice:

- With a reasoning model, `message.thinking` holds the **chain of thought** and `message.content` holds the **final answer**.
- `total_duration` is in nanoseconds: about **121 seconds**. The model generated `eval_count` = **2,614 tokens**, almost all of them thinking, and with `"stream": false` you see nothing until the very end. If you leave out `"stream": false`, the reply arrives token by token, so you can watch progress.

### Step 3: Accessing the model through Python <a id="python"></a>

Install the official Python library:

In [ ]:
# Run once if the library is missing (remove the # to enable):
# %pip install -q ollama

Next, set the model names in one place and check which of them are installed:

In [1]:
import os, re, hashlib
import ollama

CHAT_MODEL = "deepseek-r1"       # reasoning model (":latest" is the 8B version here)
FAST_MODEL = "llama3.2"          # small, fast, non-reasoning model
EMBED_MODEL = "nomic-embed-text" # embedding model, used in Part 2

installed = [m.model for m in ollama.list().models]
print("Installed models:", installed)

for name in (CHAT_MODEL, FAST_MODEL, EMBED_MODEL):
    if not any(i == name or i.startswith(name + ":") for i in installed):
        print(f"Missing: {name}  ->  run:  ollama pull {name}")

Installed models: ['nomic-embed-text:latest', 'deepseek-teacher:latest', 'llama3.2:latest', 'deepseek-r1:latest']


`ollama.chat()` takes a model name and a list of messages, and returns the reply. With `think=True` (a recent Ollama feature), a reasoning model returns its thinking in a **separate field**, so the answer is clean:

In [2]:
response = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "user", "content": "Explain Newton's second law of motion"},
    ],
    think=True,
)
print("FINAL ANSWER:\n", response["message"]["content"])

FINAL ANSWER:
 
Okay, let's break down Newton's Second Law of Motion.

**The Simple Explanation:**

Think of it like this: **Force causes acceleration, and the amount of acceleration depends on two things: how strong the force is, and how heavy the object is.**

The law is often written as:

**F = m × a**

Where:
*   **F** is the net force acting on an object (measured in Newtons, N).
*   **m** is the mass of the object (measured in kilograms, kg).
*   **a** is the acceleration of the object (measured in meters per second squared, m/s²).

**What it Means in Detail:**

1.  **Force (F) is the "Push" or "Pull":** This is the external influence that changes the motion of an object. It could be a push, a pull, gravity, friction, etc. The **net force** is the most important part – it's the sum of all forces acting on the object, considering their directions. If forces cancel each other out (like pushing equally hard left and right), the net force is zero.

2.  **Mass (m) is the "Inertia" or 

In [3]:
# The chain of thought is in a separate field (first 800 characters shown)
print("CHAIN OF THOUGHT:\n", response["message"]["thinking"][:800], "...")

CHAIN OF THOUGHT:
 
Okay, the user asked me to explain Newton's second law of motion. Let me start by recalling what I know. Newton's second law is F=ma, right? But I need to make sure I'm not missing anything. The user might be a student looking for homework help, someone curious about physics basics, or maybe a teacher preparing a lesson. I should keep it clear and educational.

First, I should break down the components: force, mass, acceleration. The user probably wants a simple explanation but with enough detail to understand. They might not be familiar with vectors, so I should mention that force and acceleration are in the same direction but note that it's a vector equation. However, for simplicity, maybe start with the scalar form.

Wait, the law also relates to the change in momentum. Maybe I should  ...


> **Reasoning models are slow.** The call above waits for the *whole* thinking section before returning anything. On a laptop this can take a minute or more, and the first call is slower because the model has to be loaded into memory. Use `ollama ps` in a terminal to check that the model is running on the GPU.

**Streaming** shows the output as it is produced, so you can watch the model think instead of staring at a blank cell:

In [4]:
stream = ollama.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Solve: (125 * 125)/5"}],
    think=True,
    stream=True,
)

phase = None
for chunk in stream:
    msg = chunk["message"]
    if getattr(msg, "thinking", None):
        if phase != "thinking":
            print("[thinking] ", end="")
            phase = "thinking"
        print(msg.thinking, end="", flush=True)
    if msg.content:
        if phase != "answer":
            print("\n\n[answer] ", end="")
            phase = "answer"
        print(msg.content, end="", flush=True)

[thinking] 
I need to solve (125 * 125) divided by 5. First, I should calculate 125 times 125. I know that 125 is 5 cubed, so 125 * 125 is 5 cubed times 5 cubed, which is 5 to the power of 6. Let me confirm that.

5^3 is 125, so 5^3 * 5^3 = 5^(3+3) = 5^6. And 5^6 is 15625. I can also think of it as 125 squared. 125 * 125.

Let me do the multiplication: 125 * 125. I can break it down. 100 * 125 is 12,500, and 25 * 125 is 3,125, so 12,500 + 3,125 = 15,625. Yeah, that's 15625.

So, 125 * 125 = 15625.

Now, I need to divide that by 5. So, 15625 / 5.

Dividing by 5 is the same as multiplying by 1/5, but I can just do the division directly.

15625 ÷ 5. Since 5 goes into 15 three times, but 15 is part of 15625.

I can think of it as dividing each part. 15625 is 15625, so dividing by 5.

5 * 3125 = 15625, because 5 * 3000 = 15,000, 5 * 125 = 625, so 15,000 + 625 = 15,625. Wait, that's the same as what I had.

125 * 125 = 15625, and 5 * 3125 = 15625, so 15625 / 5 = 3125.

But let me make sure. 

#### Setting the temperature

`options={"temperature": ...}` controls how random the output is. For a quick demo we use the small, fast `FAST_MODEL`. At temperature 0 the two runs should be (nearly) identical. At 1.2 they should differ.

In [5]:
prompt = "Write a three-sentence story about a boy and a puppy."

for temp in [0, 1.2]:
    print(f"--- temperature = {temp}")
    for _ in range(2):
        story = ollama.chat(
            model=FAST_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": temp},
        )
        print("-", story["message"]["content"].strip())

--- temperature = 0
- As the sun set over the small town, a young boy named Max sat on the porch steps, his new puppy, Luna, snuggled up beside him. With a gentle nudge, Luna encouraged Max to take her on an adventure, and together they set off into the nearby woods, exploring the winding trails and discovering hidden wonders. As the stars began to twinkle in the night sky, Max and Luna curled up together, tired but content, with the sound of crickets and rustling leaves lulling them into a peaceful sleep.
- As the sun set over the small town, a young boy named Max sat on the porch steps, his new puppy, Luna, snuggled up beside him. With a gentle nudge, Luna encouraged Max to take her on an adventure, and together they set off into the nearby woods, exploring the winding trails and discovering hidden wonders. As the stars began to twinkle in the night sky, Max and Luna curled up together, tired but content, with the sound of crickets and rustling leaves lulling them into a peaceful sle

# Part 2: A local RAG app with Gradio <a id="part2"></a>

**Retrieval-Augmented Generation (RAG)** lets a model answer questions about documents it was never trained on. Instead of hoping the model "knows" the answer, we **retrieve** the relevant pieces of the document and put them in the prompt.

We will build a demo app with Gradio that answers questions about an uploaded PDF, running entirely on your computer:

```text
 PDF -> split into chunks -> embed each chunk -> store in Chroma          (once per PDF)
 Question -> embed -> find the most similar chunks -> prompt + LLM -> Answer   (every question)
```

### Prerequisites and imports <a id="prereq"></a>

- **Python 3.10+**
- **LangChain**: framework for building LLM applications (loaders, text splitters, vector stores, retrievers)
- **Chroma**: a vector database for storing embeddings and finding similar chunks
- **Gradio**: a quick way to build a web interface
- **PyMuPDF**: reads text from PDF files

In [ ]:
# Run once if the libraries are missing (remove the # to enable):
#!pip install -q -U langchain langchain-community langchain-text-splitters langchain-ollama langchain-chroma chromadb gradio pymupdf

In [ ]:
#!pip install -U langchain_ollama

In [6]:
import gradio as gr
from langchain_classic.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma.vectorstores import Chroma
from langchain_ollama.embeddings import OllamaEmbeddings
import ollama

### Step 1: Process the PDF <a id="process"></a>

`process_pdf` loads the PDF, splits it into overlapping chunks, embeds them with `nomic-embed-text`, stores them in Chroma, and returns a **retriever** (an object that finds the chunks most relevant to a question).

Two design choices fix problems that are easy to run into:

1. **Cache by file content.** Without a cache, the whole PDF would be re-embedded for *every question*, which is slow.
2. **In-memory store, one collection per PDF.** If every upload is saved to the same `./chroma_db` folder, chunks from *earlier* PDFs pile up and get retrieved for questions about the new one.

In [7]:
_retriever_cache = {}   # file hash -> retriever, so each PDF is embedded only once

def process_pdf(pdf_path):
    """Load a PDF, split it into chunks, embed them, and return a retriever."""
    if pdf_path is None:
        return None
    pdf_path = getattr(pdf_path, "name", pdf_path)      # older Gradio versions pass a temp-file object

    with open(pdf_path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()[:32]
    if file_hash in _retriever_cache:
        return _retriever_cache[file_hash]

    data = PyMuPDFLoader(pdf_path).load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = [c for c in text_splitter.split_documents(data) if c.page_content.strip()]
    if not chunks:
        raise ValueError("No text found in this PDF. Scanned PDFs need OCR first.")

    embeddings = OllamaEmbeddings(model=EMBED_MODEL)
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=f"pdf_{file_hash}",       # in-memory, separate collection for each PDF
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})   # return the 4 most similar chunks

    _retriever_cache[file_hash] = retriever
    return retriever

In short, `process_pdf`:

- returns `None` if no PDF was uploaded
- extracts the text page by page with `PyMuPDFLoader`
- splits it into chunks of about 500 characters with 100 characters of overlap (so sentences cut at a boundary are not lost)
- generates a vector embedding for each chunk with `OllamaEmbeddings`
- stores the embeddings in Chroma for similarity search

### Step 2: Combine the retrieved chunks <a id="combine"></a>

The retriever returns a list of document objects. `combine_docs()` joins their text into a single string that can go into the prompt.

In [8]:
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

Because retrieval returns short excerpts rather than whole documents, this keeps the context readable and compact before it is passed to DeepSeek-R1.

### Step 3: Query DeepSeek-R1 using Ollama <a id="query"></a>

`ollama_llm` builds a prompt from the question and the retrieved context, and sends it to the model. The instructions tell the model to answer **only from the context** and to admit when the answer is not there. Without that instruction, a model will often fill gaps with confident guesses (hallucinations).

In [9]:
def ollama_llm(question, context, model=CHAT_MODEL):
    formatted_prompt = f"""Answer the question using only the context below. \
If the answer is not in the context, say "I don't know based on the document."

Question: {question}

Context:
{context}"""

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    response_content = response["message"]["content"]

    # Recent Ollama versions return the thinking in a separate field, so the content is already clean.
    # Older versions put it inside <think>...</think> tags, so we remove those as a safety net.
    final_answer = re.sub(r"<think>.*?</think>", "", response_content, flags=re.DOTALL).strip()
    return final_answer

The function formats the question and the retrieved context into one prompt and sends it with `ollama.chat()`. The regular expression removes any `<think>...</think>` block, and `.strip()` removes the leftover blank space around the answer.

### Step 4: The RAG pipeline <a id="pipeline"></a>

Now we connect the components: retrieve, combine, ask.

In [10]:
def rag_chain(question, retriever):
    retrieved_docs = retriever.invoke(question)
    formatted_content = combine_docs(retrieved_docs)
    return ollama_llm(question, formatted_content)

`retriever.invoke(question)` searches the vector store for the most relevant excerpts, `combine_docs` merges them, and `ollama_llm` produces an answer grounded in that text.

#### Test the pipeline without the interface (optional)

It is easier to debug the pipeline before adding the web UI. Put any PDF **with selectable text** next to this notebook, set its name below, and run the cell. It prints the retrieved chunks first, because most RAG problems are retrieval problems: if the right chunk is not retrieved, the model cannot answer.

In [11]:
from langchain_ollama import OllamaEmbeddings

In [12]:
sample_pdf = "documents/Lecture10_Intro_to_LLMs_2026_v1.pdf"       # <- change to the name of your PDF
test_question = "What is this document about?"

if os.path.exists(sample_pdf):
    retriever = process_pdf(sample_pdf)
    print("Retrieved chunks:")
    for d in retriever.invoke(test_question):
        print(f"  page {d.metadata.get('page')}: {d.page_content[:120]!r}...")
    print("\nAnswer:\n", rag_chain(test_question, retriever))
else:
    print(f"'{sample_pdf}' not found. Put a PDF next to this notebook (or skip to the Gradio app below).")

Retrieved chunks:
  page 147: '@Rahul Joglekar & Zoran B. Djordjevic\nAppendix\n148'...
  page 21: '@Rahul Joglekar & Zoran B. Djordjevic\nCommonly Used Corpora\n•\nWikipedia. Wikipedia  is an online encyclopedia containing'...
  page 36: '“Tom” can offer features associated with a person or an earlier subject.\nValue (V): what information can this token cont'...
  page 20: '•\nBooks: BookCorpus  is a commonly used dataset in previous small-scale models consisting of \nover 11,000 books covering'...

Answer:
 Based on the provided context, the document appears to be about **language models**, specifically focusing on **commonly used corpora** (collections of text) for training them. It introduces Wikipedia and several book corpora (like BookCorpus and Project Gutenberg) as examples of these text resources.

However, the context provided is just a small snippet and starts with a reference to "Appendix 148" by "@Rahul Joglekar & Zoran B. Djordjevic". This suggests that the document is li

### Step 5: Create the Gradio interface <a id="gradio"></a>

Now we wrap the pipeline in a Gradio interface that takes a PDF and a question. The function must **return the answer text itself**, so Gradio can display it, and it should give a friendly message when something is missing.

In [13]:
def ask_question(pdf_path, question):
    if pdf_path is None:
        return "Please upload a PDF first."
    if not question or not question.strip():
        return "Please type a question."
    try:
        retriever = process_pdf(pdf_path)
        return rag_chain(question, retriever)
    except Exception as e:
        return f"Error: {e}"


interface = gr.Interface(
    fn=ask_question,
    inputs=[
        gr.File(label="Upload PDF", file_types=[".pdf"]),
        gr.Textbox(label="Ask a question"),
    ],
    outputs="text",
    title="Ask questions about your PDF",
    description="Use DeepSeek-R1 to answer your questions about the uploaded PDF document.",
)

interface.launch(share=False)    # share=False keeps the app on your computer only

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


The first question about a new PDF is slower, because the PDF is embedded then. Later questions about the same file reuse the cache. A reasoning model may still take a while to answer, so be patient or switch `CHAT_MODEL` to a smaller model.

When you are finished, stop the app so the port is released:

In [ ]:
interface.close()

## Troubleshooting <a id="trouble"></a>

| Problem | Likely cause and fix |
|---|---|
| `Connection refused` / cannot connect | The Ollama app or service is not running. Start the app, or run `ollama serve` in a terminal |
| `model "..." not found` (404) | The model is not downloaded. Run `ollama pull <model>` |
| Very slow responses | The model is too big for your memory. Use a smaller tag (`deepseek-r1:1.5b`), check `ollama ps` for a high CPU share, or stream the output |
| `curl` prints nothing for a long time | `"stream": false` waits for the whole reply. Remove it to see tokens as they arrive |
| Gradio starts on port 7861 instead of 7860 | An earlier app is still running. Run `interface.close()` (or `gr.close_all()`) and launch again |
| Pydantic or Gradio schema errors | Upgrade Gradio (`pip install -U gradio`). An older workaround was pinning `pydantic==2.10.6` |
| "No text found in this PDF" | The PDF is a scan (images). It needs OCR before it can be used |
| `LangChainDeprecationWarning` for `OllamaEmbeddings` or `Chroma` | Import them from `langchain_ollama` and `langchain_chroma` instead of `langchain_community` |
| Answers mention things from an *older* PDF | The vector store was shared between uploads. Use one collection per PDF, as in `process_pdf` |

## Summary and exercises <a id="summary"></a>

| Topic | Key idea |
|---|---|
| Ollama | Runs models locally. `pull`, `list`, `run`, `ps` are the core commands |
| Reasoning models | They write a long chain of thought first. That is why they are slower |
| Modelfile | `FROM` + `PARAMETER` + `SYSTEM` customize a model. Build it with `ollama create` |
| API and Python | `message.thinking` holds the reasoning, `message.content` holds the answer. Use `stream=True` to watch progress |
| RAG | Load, chunk, embed, store, retrieve, then prompt. Tell the model to answer **only** from the context |
| Pitfalls | Cache the embeddings, keep each PDF in its own collection, and check what was retrieved before blaming the model |

**Exercises**

1. Pull `deepseek-r1:1.5b` and compare it with the 8B model on the same question, using streaming. How do speed and answer quality differ?
2. Write a Modelfile with a different temperature and system prompt. Does the behavior change?
3. Change `chunk_size` to 200 and then 1000. Which questions stop working?
4. Ask a question that is **not** answered in your PDF. Then remove the "answer only from the context" instruction from `ollama_llm` and ask again. What changes?
5. Set `CHAT_MODEL = "llama3.2"` and compare speed and answer quality with DeepSeek-R1.
6. **Challenge:** return the page numbers of the retrieved chunks (`d.metadata["page"]`) together with the answer, and show them in a second Gradio output box.

---
*This lab is adapted from a public tutorial on running DeepSeek-R1 locally with Ollama and building a Gradio RAG demo.*